In [1]:
# Session 1 — LLM Mechanics and Prompt Engineering

In [2]:
# Problem statement:

# Imagine Walmart wants to build AI applications for areas such as customer support, product listing, supply chain, incident management, supplier communication, etc.

# Before developers build those applications, they need to answer an important question:
#     “How do we control an LLM so that it gives reliable, predictable, cost-effective and structured answers suitable for a real enterprise application?”

# The notebook teaches this in two major stages.

# First: Understand how an LLM behaves internally from an engineering perspective.
# 1. Tokenization – The LLM doesn't really read sentences as humans do; it processes tokens.
# 2. Context Window – The LLM only remembers information that is supplied inside its current context.
# 3. Temperature – Controls how predictable versus creative the model's responses are.

# Second: Learn how to write prompts so the LLM performs different enterprise tasks reliably.
# 1. Zero-shot prompting → classify customer complaints without giving examples.
# 2. Few-shot prompting → give examples so the model learns exactly how Walmart wants information extracted.
# 3. Chain-of-Thought prompting → make the model reason through a supply-chain reorder decision involving stock, demand, lead time and promotion.
# 4. Role prompting → examine the same AI problem from an AI Architect, Risk Officer and Finance Lead perspective.
# 5. Structured Output prompting → convert an unstructured shipment incident into a predictable structured object using Pydantic.

In [3]:
# The final business problem in the hands-on lab:
#     Walmart receives thousands of product listings from third-party sellers. Can an LLM automatically examine each listing, classify the product correctly, detect policy violations, and decide whether the product should be approved for listing?

In [4]:
# Understand LLM → Control LLM → Prompt LLM → Structure LLM output → Apply it to a real business problem.

In [5]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

import tiktoken
# tiktoken is a tokenizer library.
# tiktoken helps us calculate approximately how many tokens a prompt contains.

load_dotenv(override=True)

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("OK Libraries loaded. OpenAI client ready.")
print(f"   Using model: gpt-4o-mini (cost-efficient for labs)")
print(f"   Using model: gpt-4o (for advanced demonstrations)")


OK Libraries loaded. OpenAI client ready.
   Using model: gpt-4o-mini (cost-efficient for labs)
   Using model: gpt-4o (for advanced demonstrations)


---
# Part 1: LLM Mechanics

Before we build production systems with LLMs, we need to understand exactly how they work — not at a research level, but at the level that matters for engineering decisions.

Three mechanics that every AI engineer must understand:
1. **Tokenization** — What the model actually reads
2. **Context Window** — What the model can remember
3. **Temperature** — How the model chooses its next word


## 1.1 Tokenization

An LLM does not read words. It reads **tokens**.

A token is roughly 3–4 characters in English. Common words are one token. Rare words, proper nouns, and code identifiers are often split into multiple tokens.

**Why this matters for engineering:**
- API costs are charged per token, not per word
- Context window limits are measured in tokens
- Poorly structured prompts waste tokens and increase cost

Let's see exactly how Walmart-relevant text gets tokenized.


In [ ]:
# Tokenization Demo
# https://platform.openai.com/tokenizer
enc = tiktoken.encoding_for_model("gpt-4o")

texts = [
    "Walmart",
    "Walmart processes 2.3 million customer transactions per hour.",
    "Supply chain optimization using LangGraph and LLM-as-judge evaluation.",
    "SKU-7829341-B",   # A product identifier — watch how this is split
    "def run_agent(state: AgentState) -> AgentState:",  # Code
]

print(f"Tokenizer: {enc.name}")
print(f"Vocabulary size: {enc.n_vocab:,} tokens")
print()
print(f"{'Text':<55} | {'Tokens':>6} | {'Token IDs (first 8)'}")
print("-" * 90)

for text in texts:
    token_ids = enc.encode(text)
    token_words = [enc.decode([t]) for t in token_ids]
    preview_ids = str(token_ids[:8]) + ("..." if len(token_ids) > 8 else "")
    print(f"{text[:54]:<55} | {len(token_ids):>6} | {preview_ids}")

print()
print("Key insight: 'SKU-7829341-B' is split into multiple tokens.")
print("Structured identifiers cost more tokens than natural text.")


Tokenizer: o200k_base
Vocabulary size: 200,019 tokens

Text                                                    | Tokens | Token IDs (first 8)
------------------------------------------------------------------------------------------
Walmart                                                 |      2 | [54, 40173]
Walmart processes 2.3 million customer transactions pe  |     13 | [54, 40173, 14340, 220, 17, 13, 18, 5749]...
Supply chain optimization using LangGraph and LLM-as-j  |     14 | [49473, 13464, 34658, 2360, 27830, 9922, 326, 451]...
SKU-7829341-B                                           |      6 | [72089, 12, 47574, 47717, 16, 8287]
def run_agent(state: AgentState) -> AgentState:         |     12 | [1314, 2461, 62035, 16114, 25, 28237, 1881, 8]...

Key insight: 'SKU-7829341-B' is split into multiple tokens.
Structured identifiers cost more tokens than natural text.


### Token Cost Calculator

At $0.150 per 1M input tokens (gpt-4o-mini), even small inefficiencies in prompts add up at Walmart's scale.


In [7]:
# Token Cost Calculator
def estimate_cost(text: str, model: str = "gpt-4o-mini", direction: str = "input") -> dict:
    """Estimate the token count and API cost for a given text."""
    enc = tiktoken.encoding_for_model("gpt-4o")  # gpt-4o enc works for gpt-4o-mini too
    token_count = len(enc.encode(text)) # "Hello Walmart" -> [13225, 54321]
    
    # Pricing as of 2026 (approximate, verify at platform.openai.com)
    pricing = {
        "gpt-4o-mini": {"input": 0.150, "output": 0.600},   # per 1M tokens
        "gpt-4o":      {"input": 2.50,  "output": 10.00},
    }
    
    price_per_million = pricing.get(model, {}).get(direction, 0)
    cost_usd = (token_count / 1_000_000) * price_per_million
    
    return {
        "text_chars": len(text),
        "token_count": token_count,
        "chars_per_token": round(len(text) / token_count, 2) if token_count > 0 else 0,
        "cost_usd": round(cost_usd, 8),
        "cost_per_1000_calls": round(cost_usd * 1000, 4),
    }

# Compare prompt efficiency
prompts = {
    "Verbose system prompt": """You are an AI assistant working at Walmart. Your job is to help customer service 
    representatives answer questions. You should always be helpful, professional, and accurate. You must never 
    make up information that you do not know. Always be polite and courteous to customers.""",
    
    "Concise system prompt": """You are a Walmart customer service AI. Be accurate, professional, brief.""",
}

print("Prompt Efficiency Comparison (gpt-4o-mini, input tokens):")
print("-" * 70)
for name, prompt in prompts.items():
    stats = estimate_cost(prompt, "gpt-4o-mini", "input")
    print(f"\n{name}:")
    print(f"  Characters : {stats['text_chars']}")
    print(f"  Tokens     : {stats['token_count']}")
    print(f"  Cost/call  : ${stats['cost_usd']:.6f}")
    print(f"  Cost/1000 calls: ${stats['cost_per_1000_calls']}")

print()
print("At 1M calls/day (Walmart scale), the difference compounds significantly.")


Prompt Efficiency Comparison (gpt-4o-mini, input tokens):
----------------------------------------------------------------------

Verbose system prompt:
  Characters : 284
  Tokens     : 55
  Cost/call  : $0.000008
  Cost/1000 calls: $0.0083

Concise system prompt:
  Characters : 72
  Tokens     : 15
  Cost/call  : $0.000002
  Cost/1000 calls: $0.0023

At 1M calls/day (Walmart scale), the difference compounds significantly.


## 1.2 Context Window

The context window is the model's working memory — everything it can "see" at once.

| Model | Context Window |
|---|---|
| gpt-4o-mini | 128,000 tokens |
| gpt-4o | 128,000 tokens |
| Claude Sonnet | 200,000 tokens |

**Why this matters:**
- Long documents must be chunked if they exceed the window
- Every message in a chat adds to the context and costs tokens
- Agents with long tool call histories can hit context limits mid-task

Let's demonstrate how context window enables multi-turn memory.


In [ ]:
# Context Window Demo — Multi-turn Memory
print("=== Demo: How Context Window Creates Memory ===\n")

# Conversation 1: Without context history (model has no memory)
# First scenario: ask without previous conversation history
print("NO Without context (no message history passed):")
response_no_context = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "What inventory threshold did I mention?"}
    ],
    max_tokens=80,
    temperature=0
)
print(f"   Model says: {response_no_context.choices[0].message.content.strip()}\n")

# Conversation 2: With full context history passed
# Second scenario: provide the full conversation history
print("OK With context (full message history passed):")
conversation_history = [
    {"role": "user",      "content": "Our reorder threshold for SKU-7829 is 500 units."},
    {"role": "assistant", "content": "Understood. The reorder threshold for SKU-7829 is 500 units."},
    {"role": "user",      "content": "And for SKU-9102 it's 250 units."},
    {"role": "assistant", "content": "Got it. SKU-9102 reorder threshold is 250 units."},
    {"role": "user",      "content": "What inventory threshold did I mention for SKU-7829?"},
]

response_with_context = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=conversation_history,
    max_tokens=80,
    temperature=0
)
print(f"   Model says: {response_with_context.choices[0].message.content.strip()}\n")

# Show context size
enc = tiktoken.encoding_for_model("gpt-4o")
total_tokens = sum(len(enc.encode(m["content"])) for m in conversation_history)
print(f"STATS Context used: ~{total_tokens} tokens")
print(f"   Window capacity: 128,000 tokens")
print(f"   Remaining: {128000 - total_tokens:,} tokens")


=== Demo: How Context Window Creates Memory ===

NO Without context (no message history passed):
   Model says: I'm sorry, but I don't have access to previous conversations or specific details about your inventory threshold. If you provide me with more context or details, I would be happy to help you with your inquiry!

OK With context (full message history passed):
   Model says: You mentioned that the inventory reorder threshold for SKU-7829 is 500 units.

STATS Context used: ~66 tokens
   Window capacity: 128,000 tokens
   Remaining: 127,934 tokens


## 1.3 Temperature

Temperature controls **randomness** in the model's output.

| Temperature | Behaviour | Use Case |
|---|---|---|
| 0.0 | Deterministic, always most likely token | Classification, extraction, code generation |
| 0.2–0.5 | Slightly varied, mostly consistent | Summaries, factual Q&A, structured outputs |
| 0.7–1.0 | Creative, more varied | Marketing copy, ideation, creative writing |
| > 1.2 | Highly random, may be incoherent | Rarely useful in production |

**Rule of thumb for production systems:** Use the lowest temperature that produces acceptable output quality.


In [10]:
# Temperature Demo
import time

prompt = "In one sentence, suggest how Walmart can use AI to improve store checkout speed."

print("=== Temperature Effect on Output Variability ===\n")
print(f"Prompt: \"{prompt}\"\n")

for temp in [0.0, 0.5, 1.2, 1.8]:
    print(f"Temperature = {temp}:")
    responses = []
    for i in range(3):
        r = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
            max_tokens=60,
        )
        responses.append(r.choices[0].message.content.strip())
        time.sleep(0.3)  # Avoid rate limiting
    
    for i, resp in enumerate(responses, 1):
        print(f"  Run {i}: {resp}")
    print()

print("Observation: At temp=0.0, all 3 runs give identical output.")
print("At temp=1.2, outputs diverge significantly — unpredictable for production.")


=== Temperature Effect on Output Variability ===

Prompt: "In one sentence, suggest how Walmart can use AI to improve store checkout speed."

Temperature = 0.0:
  Run 1: Walmart can implement AI-powered self-checkout systems that utilize computer vision and machine learning to streamline the scanning process and reduce wait times for customers.
  Run 2: Walmart can implement AI-powered checkout systems that utilize computer vision and machine learning to automatically scan and process items in customers' carts, significantly reducing wait times at checkout.
  Run 3: Walmart can implement AI-powered checkout systems that utilize computer vision and machine learning to automatically scan and process items in customers' carts, significantly reducing wait times at checkout.

Temperature = 0.5:
  Run 1: Walmart can implement AI-powered checkout systems that utilize computer vision and machine learning to automatically scan and process items in shoppers' carts, significantly reducing wait ti

---
# Part 2: Prompt Engineering

Prompt engineering is the practice of designing inputs to LLMs to consistently produce the outputs you need.

It is not a workaround. It is a core engineering discipline. A well-engineered prompt can often outperform a fine-tuned model — at a fraction of the cost.

We will cover 5 patterns, each with a Walmart business use case.


## 2.1 Zero-Shot Prompting

Ask the model to perform a task with no examples. Works well when:
- The task is common in the model's training data
- The output format is simple

**Walmart use case:** Classify the urgency of a customer support ticket.


In [11]:
# Zero-Shot Prompting
def zero_shot_classify(complaint: str) -> str:
    """Classify complaint severity with no examples — pure zero-shot."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a Walmart customer support triage system. "
                    "Classify each complaint as CRITICAL, HIGH, MEDIUM, or LOW severity. "
                    "Reply with exactly one word: the severity level."
                )
            },
            {"role": "user", "content": f"Complaint: {complaint}"}
        ],
        temperature=0,
        max_tokens=10,
    )
    return response.choices[0].message.content.strip()

complaints = [
    "My item arrived damaged and I need a replacement immediately.",
    "Website was slow this morning but seems fine now.",
    "I was charged twice for the same order — my bank is disputing it.",
    "The packaging could be more eco-friendly.",
    "I can't log into my Walmart+ account and my groceries won't deliver.",
]

print("Zero-Shot Complaint Classification:")
print("-" * 65)
for c in complaints:
    severity = zero_shot_classify(c)
    emoji = {"CRITICAL": "CRITICAL", "HIGH": "HIGH", "MEDIUM": "MEDIUM", "LOW": "LOW"}.get(severity, "UNKNOWN")
    print(f"{emoji} {severity:<10} | {c[:55]}")


Zero-Shot Complaint Classification:
-----------------------------------------------------------------
CRITICAL CRITICAL   | My item arrived damaged and I need a replacement immedi
LOW LOW        | Website was slow this morning but seems fine now.
CRITICAL CRITICAL   | I was charged twice for the same order — my bank is dis
LOW LOW        | The packaging could be more eco-friendly.
CRITICAL CRITICAL   | I can't log into my Walmart+ account and my groceries w


## 2.2 Few-Shot Prompting

Provide examples of input → output pairs before the actual task. The model learns the pattern from your examples.

**When to use over zero-shot:**
- You need a specific output format
- The task has domain-specific nuances the model might not know
- Zero-shot gives inconsistent results

**Walmart use case:** Extract structured data from unstructured customer feedback.


In [12]:
# Few-Shot Prompting
def few_shot_extract(feedback: str) -> dict:
    """Extract structured data from feedback using few-shot examples."""
    
    few_shot_prompt = """Extract product information from customer feedback.
Return a JSON object with keys: product_name, sentiment (positive/negative/neutral), issue (or null), rating_implied (1-5).
Return ONLY valid JSON. Do not include markdown or extra text.

Example 1:
Feedback: "The Great Value coffee maker stopped working after 3 weeks. Very disappointing."
Output: {{"product_name": "Great Value coffee maker", "sentiment": "negative", "issue": "stopped working after 3 weeks", "rating_implied": 1}}

Example 2:
Feedback: "Absolutely love my new Onn. TV! Picture quality is stunning and setup was easy."
Output: {{"product_name": "Onn. TV", "sentiment": "positive", "issue": null, "rating_implied": 5}}

Example 3:
Feedback: "The Sam's Choice pasta is decent. Nothing special but good value for the price."
Output: {{"product_name": "Sam's Choice pasta", "sentiment": "neutral", "issue": null, "rating_implied": 3}}

Now extract from:
Feedback: "{feedback}"
Output:"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": few_shot_prompt.format(feedback=feedback)}],
        response_format={"type": "json_object"},
        temperature=0,
        max_tokens=120,
    )
    
    raw = response.choices[0].message.content.strip()
    return json.loads(raw)

feedbacks = [
    "The Equate pain relief tablets work great — I use them every day and they're half the price of brand name.",
    "My Walmart+ order was 2 hours late and the frozen food was partially thawed. Very frustrated.",
    "George men's jeans fit perfectly and the material feels durable. Will buy again.",
]

print("Few-Shot Structured Extraction:")
print("=" * 65)
for fb in feedbacks:
    result = few_shot_extract(fb)
    print(f"\nFeedback: {fb[:60]}...")
    print(f"Extracted: {json.dumps(result, indent=2)}")

Few-Shot Structured Extraction:

Feedback: The Equate pain relief tablets work great — I use them every...
Extracted: {
  "product_name": "Equate pain relief tablets",
  "sentiment": "positive",
  "issue": null,
  "rating_implied": 5
}

Feedback: My Walmart+ order was 2 hours late and the frozen food was p...
Extracted: {
  "product_name": "Walmart+ order",
  "sentiment": "negative",
  "issue": "order was 2 hours late and frozen food was partially thawed",
  "rating_implied": 1
}

Feedback: George men's jeans fit perfectly and the material feels dura...
Extracted: {
  "product_name": "George men's jeans",
  "sentiment": "positive",
  "issue": null,
  "rating_implied": 5
}


## 2.3 Chain-of-Thought (CoT) Prompting

Tell the model to **think step by step** before giving a final answer. This forces the model to reason through multi-step problems rather than pattern-match to the nearest answer.

**When to use:**
- Multi-step reasoning (math, logic, planning)
- Decisions that require considering multiple factors
- Any situation where you'd want a human to "show their work"

**Walmart use case:** Supply chain reorder decision with multiple constraints.


In [13]:
# Chain-of-Thought Prompting

# ── WITHOUT Chain-of-Thought ──
no_cot_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": """
A Walmart distribution center has the following situation:
- Current stock of Product X: 800 units
- Average daily demand: 65 units/day
- Lead time to restock: 8 days
- Safety stock required: 4 days of demand
- Upcoming promotional event in 5 days expected to triple demand for 3 days

Should we place a reorder today? Answer yes or no.
"""}
    ],
    temperature=0,
    max_tokens=20,
)
print("WITHOUT Chain-of-Thought:")
print(f"  Answer: {no_cot_response.choices[0].message.content.strip()}\n")

# ── WITH Chain-of-Thought ──
cot_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": """
A Walmart distribution center has the following situation:
- Current stock of Product X: 800 units
- Average daily demand: 65 units/day
- Lead time to restock: 8 days
- Safety stock required: 4 days of demand
- Upcoming promotional event in 5 days expected to triple demand for 3 days

Should we place a reorder today? Think step by step, calculate each quantity explicitly, then give a final recommendation with justification.
"""}
    ],
    temperature=0,
    max_tokens=400,
)
print("WITH Chain-of-Thought:")
print(cot_response.choices[0].message.content.strip())


WITHOUT Chain-of-Thought:
  Answer: To determine whether to place a reorder today, we need to calculate the stock situation considering the upcoming promotional

WITH Chain-of-Thought:
To determine whether to place a reorder for Product X today, we need to analyze the current stock, demand, lead time, safety stock, and the impact of the upcoming promotional event. Let's break this down step by step.

### Step 1: Calculate the Safety Stock
Safety stock is calculated based on the average daily demand and the number of days of safety stock required.

- **Average daily demand**: 65 units/day
- **Safety stock required**: 4 days of demand

\[
\text{Safety Stock} = \text{Average Daily Demand} \times \text{Safety Stock Days} = 65 \, \text{units/day} \times 4 \, \text{days} = 260 \, \text{units}
\]

### Step 2: Calculate the Reorder Point
The reorder point is the level of inventory at which a new order should be placed. It is calculated as the sum of the safety stock and the demand during the l

## 2.4 Role Prompting

Define a specific persona and expertise for the model. This activates relevant knowledge and adjusts the communication style to match the role.

**When to use:**
- You need domain-specific language and framing
- The audience for the output is specific (executives vs engineers vs customers)
- Tone and perspective matter

**Walmart use case:** Get different analyses of the same problem from different roles.


In [14]:
# Role Prompting — Same Question, Different Roles

business_problem = """
Walmart is considering deploying an AI agent to automate supplier negotiation emails. 
The agent would draft, send, and respond to routine supplier communications autonomously.
What are the key considerations?
"""

roles = {
    "AI Architect": "You are a Senior AI Systems Architect at Walmart Global Tech with 15 years of experience in enterprise AI deployments. Focus on technical architecture, reliability, and system design.",
    "Risk Officer": "You are the Chief Risk Officer at a Fortune 500 retailer. Focus on legal liability, compliance, audit trails, and reputational risk.",
    "Finance Lead": "You are VP of Finance at Walmart. Focus on ROI, cost savings, implementation costs, and financial governance.",
}

for role_name, system_prompt in roles.items():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": business_problem},
        ],
        temperature=0.2,
        max_tokens=200,
    )
    print(f"\n{'=' * 65}")
    print(f"Role: {role_name}")
    print('=' * 65)
    print(response.choices[0].message.content.strip())



Role: AI Architect
Deploying an AI agent to automate supplier negotiation emails involves several key considerations that span technical architecture, reliability, compliance, and user experience. Here are the critical factors to consider:

### 1. **Natural Language Processing (NLP) Capabilities**
   - **Understanding Context**: The AI must accurately interpret the context of negotiations, including pricing, terms, and supplier relationships.
   - **Language Proficiency**: Ensure the AI can communicate effectively in the languages used by suppliers, including industry-specific jargon.
   - **Sentiment Analysis**: Implement sentiment analysis to gauge the tone of supplier responses and adjust communication strategies accordingly.

### 2. **Data Security and Compliance**
   - **Data Privacy**: Ensure compliance with data protection regulations (e.g., GDPR, CCPA) when handling supplier information.
   - **Confidentiality**: Implement measures to protect sensitive negotiation details and 

## 2.5 Structured Output Prompting

Force the model to return machine-readable JSON that conforms to a specific schema. In production, this eliminates fragile string parsing.

**The modern approach:** Use OpenAI's native structured output with Pydantic models — guaranteed schema compliance.

**Walmart use case:** Extract a shipment incident report from a free-text description.


In [15]:
from pydantic import BaseModel, Field
from typing import Optional, List
from enum import Enum

# Define the exact schema we want
class SeverityLevel(str, Enum):
    CRITICAL = "CRITICAL"
    HIGH = "HIGH"
    MEDIUM = "MEDIUM"
    LOW = "LOW"

class ShipmentIncidentReport(BaseModel):
    incident_id: str = Field(description="Generate a short unique ID like INC-001")
    sku: Optional[str] = Field(description="Product SKU if mentioned, else null")
    incident_type: str = Field(description="Category: damaged_goods, delayed_delivery, wrong_item, missing_item, other")
    severity: SeverityLevel
    financial_impact_usd: Optional[float] = Field(description="Estimated dollar impact if mentioned, else null")
    requires_immediate_action: bool
    recommended_actions: List[str] = Field(description="List of 2-3 concrete next steps")
    summary: str = Field(description="One sentence summary of the incident")

# Parse a free-text incident description
incident_text = """
Our night shift supervisor just called in. A full pallet of SKU-8832-A (Equate Vitamins, 500 count) 
got soaked during the storm because the loading dock door was left open. About 240 units are 
completely unsaleable. At $12 retail each, that's roughly $2,880 in damaged inventory. 
The supplier needs to be notified and we probably need an emergency restock request since 
we're already low on this SKU ahead of the weekend rush.
"""

response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a Walmart operations AI. Extract structured incident information from descriptions."},
        {"role": "user", "content": f"Parse this incident:\n\n{incident_text}"},
    ],
    response_format=ShipmentIncidentReport,
    temperature=0,
)

report = response.choices[0].message.parsed
print("Structured Incident Report:")
print("=" * 50)
print(f"Incident ID      : {report.incident_id}")
print(f"SKU              : {report.sku}")
print(f"Type             : {report.incident_type}")
print(f"Severity         : {report.severity.value}")
print(f"Financial Impact : ${report.financial_impact_usd:,.2f}" if report.financial_impact_usd else "Financial Impact : Not specified")
print(f"Immediate Action : {'WARNING  YES' if report.requires_immediate_action else 'No'}")
print(f"Summary          : {report.summary}")
print(f"\nRecommended Actions:")
for i, action in enumerate(report.recommended_actions, 1):
    print(f"  {i}. {action}")
print()
print("OK Structured output: guaranteed schema, no parsing errors.")


Structured Incident Report:
Incident ID      : INC-001
SKU              : SKU-8832-A
Type             : damaged_goods
Severity         : HIGH
Financial Impact : $2,880.00
Immediate Action : WARNING  YES
Summary          : A full pallet of Equate Vitamins got soaked during the storm due to an open loading dock door.

Recommended Actions:
  1. Notify the supplier about the damaged goods
  2. Submit an emergency restock request for SKU-8832-A
  3. Inspect the loading dock to prevent future incidents

OK Structured output: guaranteed schema, no parsing errors.


In [16]:
print(report)

incident_id='INC-001' sku='SKU-8832-A' incident_type='damaged_goods' severity=<SeverityLevel.HIGH: 'HIGH'> financial_impact_usd=2880.0 requires_immediate_action=True recommended_actions=['Notify the supplier about the damaged goods', 'Submit an emergency restock request for SKU-8832-A', 'Inspect the loading dock to prevent future incidents'] summary='A full pallet of Equate Vitamins got soaked during the storm due to an open loading dock door.'


---
# Session Recap

| Concept | Key Takeaway |
|---|---|
| **Tokenization** | LLMs read tokens, not words. Cost and limits are token-based. |
| **Context Window** | The model's working memory. Everything passed in the messages list is "seen". |
| **Temperature** | Use 0 for deterministic tasks, 0.2-0.5 for summaries, higher only for creativity. |
| **Zero-Shot** | Works well for common tasks. No examples needed. |
| **Few-Shot** | Add examples when zero-shot is inconsistent or needs a specific format. |
| **Chain-of-Thought** | Force step-by-step reasoning for multi-step problems. |
| **Role Prompting** | Activate domain knowledge and adjust tone by assigning a persona. |
| **Structured Output** | Use Pydantic + `beta.chat.completions.parse` for guaranteed JSON schemas. |